<a href="https://colab.research.google.com/github/18217265596/sx/blob/master/LigandMPNN_Colab_Complete_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LigandMPNN — Google Colab 完整运行版 v3

本 Notebook 包含：

1. 检查 Colab GPU 和 Python 环境  
2. 克隆 LigandMPNN 官方仓库  
3. 安装适配当前 Colab 的依赖  
4. 下载 LigandMPNN 与 ProteinMPNN 权重  
5. 修复 PyTorch 2.6+ 的 `torch.load(weights_only=True)` 兼容问题  
6. 验证 checkpoint  
7. 运行官方 `1BC8.pdb` LigandMPNN demo  
8. 上传自己的 PDB，选择模型、链和固定/重设计残基  
9. 显示 FASTA、统计结果并打包下载  

在 Colab 中先选择：

**运行时 → 更改运行时类型 → T4 GPU**

然后从上到下依次运行单元格。

> 模型选择：
>
> - `ligand_mpnn`：设计时需要利用小分子、金属、辅因子、核酸等非蛋白原子。
> - `protein_mpnn`：纯蛋白骨架、蛋白–蛋白复合物或 RFdiffusion binder 设计。

In [ ]:
# 0. 检查当前运行时
import os
import sys
import platform
import torch

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime reported by PyTorch:", torch.version.cuda)
else:
    print(
        "\n警告：当前没有 GPU。LigandMPNN 可以在 CPU 上运行，"
        "但建议在“运行时 → 更改运行时类型”中选择 T4 GPU。"
    )

Python executable: /usr/bin/python3
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA runtime reported by PyTorch: 12.8


In [ ]:
# 1. 克隆官方仓库并安装 Colab 兼容依赖
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path("/content/LigandMPNN")
RESET_REPOSITORY = True

if RESET_REPOSITORY and ROOT.exists():
    shutil.rmtree(ROOT)

if not (ROOT / "run.py").exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/dauparas/LigandMPNN.git",
            str(ROOT),
        ],
        check=True,
    )

# 不安装官方 requirements.txt：
# 它会固定旧版 torch、CUDA wheel、NumPy 和 ProDy，可能覆盖 Colab 自带环境。
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "ProDy==2.6.1",
        "biopython>=1.81",
        "ml-collections==0.1.1",
        "dm-tree==0.1.8",
    ],
    check=True,
)

print("Repository:", ROOT)
print("Current commit:")
subprocess.run(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"],
    check=True,
)

Repository: /content/LigandMPNN
Current commit:


CompletedProcess(args=['git', '-C', '/content/LigandMPNN', 'log', '-1', '--oneline'], returncode=0)

In [ ]:
# 2. 下载 LigandMPNN 和 ProteinMPNN 权重
from pathlib import Path
import subprocess

MODEL_DIR = ROOT / "model_params"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    "ligand_mpnn_010": {
        "path": MODEL_DIR / "ligandmpnn_v_32_010_25.pt",
        "url": (
            "https://files.ipd.uw.edu/pub/ligandmpnn/"
            "ligandmpnn_v_32_010_25.pt"
        ),
    },
    "protein_mpnn_020": {
        "path": MODEL_DIR / "proteinmpnn_v_48_020.pt",
        "url": (
            "https://files.ipd.uw.edu/pub/ligandmpnn/"
            "proteinmpnn_v_48_020.pt"
        ),
    },
}

def download_weight(path: Path, url: str) -> None:
    # 小于 1 MiB 通常意味着下载失败、错误页或不完整文件。
    if path.exists() and path.stat().st_size >= 1024**2:
        print(
            f"Already present: {path.name} "
            f"({path.stat().st_size / 1024**2:.1f} MiB)"
        )
        return

    path.unlink(missing_ok=True)
    subprocess.run(
        [
            "wget",
            "--show-progress",
            "--tries=4",
            "--timeout=30",
            "-O", str(path),
            url,
        ],
        check=True,
    )

    if not path.exists() or path.stat().st_size < 1024**2:
        raise RuntimeError(
            f"权重下载异常：{path}。请重新运行该单元格。"
        )

    print(
        f"Downloaded: {path.name} "
        f"({path.stat().st_size / 1024**2:.1f} MiB)"
    )

for item in WEIGHTS.values():
    download_weight(item["path"], item["url"])

Downloaded: ligandmpnn_v_32_010_25.pt (10.1 MiB)
Downloaded: proteinmpnn_v_48_020.pt (6.4 MiB)


In [ ]:
# 3. 修复 PyTorch 2.6+ checkpoint 加载兼容性
import os
from pathlib import Path

# 对未显式传入 weights_only 的 torch.load 生效；
# subprocess 会继承该环境变量。
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

run_py = ROOT / "run.py"
source = run_py.read_text(encoding="utf-8")

replacements = {
    "torch.load(checkpoint_path, map_location=device)": (
        "torch.load("
        "checkpoint_path, map_location=device, weights_only=False"
        ")"
    ),
    "torch.load(args.checkpoint_path_sc, map_location=device)": (
        "torch.load("
        "args.checkpoint_path_sc, map_location=device, "
        "weights_only=False"
        ")"
    ),
}

for old, new in replacements.items():
    source = source.replace(old, new)

run_py.write_text(source, encoding="utf-8")

print("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD =",
      os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"])
print("\n当前 run.py 中的 torch.load：")
for line in run_py.read_text(encoding="utf-8").splitlines():
    if "torch.load(" in line:
        print(" ", line.strip())

# 确保主 checkpoint 那一处确实已修复。
patched_source = run_py.read_text(encoding="utf-8")
if (
    "checkpoint_path, map_location=device, weights_only=False"
    not in patched_source
):
    raise RuntimeError("run.py 的主 checkpoint 加载位置未成功修复。")

TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD = 1

当前 run.py 中的 torch.load：
  checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
  checkpoint_sc = torch.load(args.checkpoint_path_sc, map_location=device, weights_only=False)


In [ ]:
# 4. 验证依赖和两个 checkpoint
import os
import sys
import torch
import numpy as np
import prody
import Bio

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("ProDy:", prody.__version__)
print("Biopython:", Bio.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for label, item in WEIGHTS.items():
    path = item["path"]
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )
    print("\n", label)
    print(" path:", path)
    print(" size:", f"{path.stat().st_size / 1024**2:.1f} MiB")
    print(" keys:", sorted(checkpoint.keys()))
    print(" num_edges:", checkpoint.get("num_edges"))
    print(" atom_context_num:", checkpoint.get("atom_context_num"))

Python: 3.12.13
PyTorch: 2.11.0+cu128
NumPy: 2.0.2
ProDy: 2.6.1
Biopython: 1.88
CUDA available: True
GPU: Tesla T4

 ligand_mpnn_010
 path: /content/LigandMPNN/model_params/ligandmpnn_v_32_010_25.pt
 size: 10.1 MiB
 keys: ['atom_context_num', 'model_state_dict', 'noise_level', 'num_edges']
 num_edges: 32
 atom_context_num: 25

 protein_mpnn_020
 path: /content/LigandMPNN/model_params/proteinmpnn_v_48_020.pt
 size: 6.4 MiB
 keys: ['model_state_dict', 'noise_level', 'num_edges']
 num_edges: 48
 atom_context_num: None


## A. 官方 LigandMPNN demo

使用官方仓库自带的 `inputs/1BC8.pdb`，运行真正的 `ligand_mpnn` 模型。

当前参数：

- `batch_size = 2`
- `number_of_batches = 2`
- 共生成 4 条设计序列

In [ ]:
# 5. 定义一个会显示完整 stdout/stderr 的运行函数
import os
import subprocess
from pathlib import Path
from typing import Sequence

def run_and_show(
    command: Sequence[str],
    cwd: Path = ROOT,
) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    env["PYTHONUNBUFFERED"] = "1"

    print("Running command:\n")
    print(" ".join(map(str, command)))
    print("\n" + "=" * 90)

    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout)
    print("=" * 90)
    print("Exit status:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError(
            "LigandMPNN 运行失败。完整报错已显示在上方，"
            "请不要只复制最下面的 RuntimeError。"
        )

    return result

In [ ]:
# 修复 OpenFold 与新版 NumPy 的兼容性，然后重新运行官方 LigandMPNN demo
import re
import shutil
import sys
from pathlib import Path

# 修复 NumPy 1.24+ 已删除的旧类型别名
numpy_aliases = {
    r"\bnp\.int\b": "int",
    r"\bnp\.float\b": "float",
    r"\bnp\.bool\b": "bool",
    r"\bnp\.object\b": "object",
    r"\bnp\.str\b": "str",
    r"\bnp\.complex\b": "complex",
}

changed_files = []

for py_file in ROOT.rglob("*.py"):
    source = py_file.read_text(encoding="utf-8")
    patched = source

    for pattern, replacement in numpy_aliases.items():
        patched = re.sub(pattern, replacement, patched)

    if patched != source:
        py_file.write_text(patched, encoding="utf-8")
        changed_files.append(py_file.relative_to(ROOT))

print("NumPy compatibility patch completed.")
print("Changed files:")
for path in changed_files:
    print(" ", path)

DEMO_PDB = ROOT / "inputs" / "1BC8.pdb"
DEMO_OUT = ROOT / "outputs" / "colab_ligand_demo"

if not DEMO_PDB.exists():
    raise FileNotFoundError(f"官方 demo PDB 不存在：{DEMO_PDB}")

shutil.rmtree(DEMO_OUT, ignore_errors=True)

demo_command = [
    sys.executable,
    "-u",
    "run.py",
    "--model_type",
    "ligand_mpnn",
    "--checkpoint_ligand_mpnn",
    str(WEIGHTS["ligand_mpnn_010"]["path"]),
    "--seed",
    "111",
    "--pdb_path",
    str(DEMO_PDB),
    "--out_folder",
    str(DEMO_OUT),
    "--batch_size",
    "2",
    "--number_of_batches",
    "2",
    "--temperature",
    "0.10",
    "--verbose",
    "1",
]

run_and_show(demo_command)

NumPy compatibility patch completed.
Changed files:
  openfold/data/templates.py
  openfold/np/residue_constants.py
  openfold/np/relax/utils.py
Running command:

/usr/bin/python3 -u run.py --model_type ligand_mpnn --checkpoint_ligand_mpnn /content/LigandMPNN/model_params/ligandmpnn_v_32_010_25.pt --seed 111 --pdb_path /content/LigandMPNN/inputs/1BC8.pdb --out_folder /content/LigandMPNN/outputs/colab_ligand_demo --batch_size 2 --number_of_batches 2 --temperature 0.10 --verbose 1

Designing protein from this path: /content/LigandMPNN/inputs/1BC8.pdb
These residues will be redesigned:  ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26', 'C27', 'C28', 'C29', 'C30', 'C31', 'C32', 'C33', 'C34', 'C35', 'C36', 'C37', 'C38', 'C39', 'C40', 'C41', 'C42', 'C43', 'C44', 'C45', 'C46', 'C47', 'C48', 'C49', 'C50', 'C51', 'C52', 'C53', 'C54', 'C55', 'C56', 'C57', 'C58', 'C59', 'C60

CompletedProcess(args=['/usr/bin/python3', '-u', 'run.py', '--model_type', 'ligand_mpnn', '--checkpoint_ligand_mpnn', '/content/LigandMPNN/model_params/ligandmpnn_v_32_010_25.pt', '--seed', '111', '--pdb_path', '/content/LigandMPNN/inputs/1BC8.pdb', '--out_folder', '/content/LigandMPNN/outputs/colab_ligand_demo', '--batch_size', '2', '--number_of_batches', '2', '--temperature', '0.10', '--verbose', '1'], returncode=0, stdout="Designing protein from this path: /content/LigandMPNN/inputs/1BC8.pdb\nThese residues will be redesigned:  ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26', 'C27', 'C28', 'C29', 'C30', 'C31', 'C32', 'C33', 'C34', 'C35', 'C36', 'C37', 'C38', 'C39', 'C40', 'C41', 'C42', 'C43', 'C44', 'C45', 'C46', 'C47', 'C48', 'C49', 'C50', 'C51', 'C52', 'C53', 'C54', 'C55', 'C56', 'C57', 'C58', 'C59', 'C60', 'C61', 'C62', 'C63', 'C64', 'C65', 'C66', 'C67', 'C

In [ ]:
# 7. 查看官方 demo 的输出文件和 FASTA
from pathlib import Path

demo_files = sorted(
    p for p in DEMO_OUT.rglob("*")
    if p.is_file()
)

print("Output files:")
for path in demo_files:
    print(" ", path.relative_to(DEMO_OUT))

demo_fastas = sorted((DEMO_OUT / "seqs").glob("*.fa"))

if not demo_fastas:
    raise FileNotFoundError(
        f"没有在 {DEMO_OUT / 'seqs'} 中找到 FASTA。"
    )

for fasta in demo_fastas:
    print("\n" + "=" * 90)
    print(fasta.name)
    print("=" * 90)
    print(fasta.read_text(encoding="utf-8"))

Output files:
  backbones/1BC8_1.pdb
  backbones/1BC8_2.pdb
  backbones/1BC8_3.pdb
  backbones/1BC8_4.pdb
  seqs/1BC8.fa

1BC8.fa
>1BC8, T=0.1, seed=111, num_res=93, num_ligand_res=41, use_ligand_context=True, ligand_cutoff_distance=8.0, batch_size=2, number_of_batches=2, model_path=/content/LigandMPNN/model_params/ligandmpnn_v_32_010_25.pt
MDSAITLWQFLLQLLQKPQNKHMICWTSNDGQFKLLQAEEVARLWGIRKNKPNMNYDKLSRALRYYYVKNIIKKVNGQKFVYKFVSYPEILNM
>1BC8, id=1, T=0.1, seed=111, overall_confidence=0.4797, ligand_confidence=0.5401, seq_rec=0.4731
GRSPISLHEFILELLSKPEYEGIIKWTSDDGEFRFVDPEAVAKLWGEVKGKPKMNWKNLHRALRGYKKKKIIETVKDKPYQYRFVNYPEHLHH
>1BC8, id=2, T=0.1, seed=111, overall_confidence=0.4697, ligand_confidence=0.5211, seq_rec=0.5054
MRSPISLHEFILELLSDPAYAGIIRWTSDDGRFQLVDPEAVAKLWGEEKGKPKMNWKNLHRALRGYKKKKIIKTVKGKPYQYQFVNYPELLHH
>1BC8, id=3, T=0.1, seed=111, overall_confidence=0.4774, ligand_confidence=0.5428, seq_rec=0.4839
MKSPISLHEFLLRLLSDPAYADIIEWVSDNGEFRLVDPEAVAKLWGEEKGKPKMNWKNLHRALRGYKKKKIIETVKGKPYT

## B. 上传并设计自己的 PDB

下面一次上传一个 PDB。上传后会简单列出：

- 蛋白链 ID
- 每条链的 ATOM 残基数
- PDB 中的 HETATM residue names

对于 RFdiffusion 生成的 binder–target 复合物：

- 通常选择 `MODEL_TYPE = "protein_mpnn"`
- `CHAINS_TO_DESIGN` 填 binder 链，如 `"A"`
- target 链不会被改动

In [ ]:
# 8. 上传一个自己的 PDB
from google.colab import files
from pathlib import Path

uploaded = files.upload()

pdb_items = [
    (name, data)
    for name, data in uploaded.items()
    if name.lower().endswith(".pdb")
]

if len(pdb_items) != 1:
    raise ValueError("请一次只上传一个 .pdb 文件。")

original_name, pdb_bytes = pdb_items[0]

USER_INPUT_DIR = ROOT / "user_inputs"
USER_INPUT_DIR.mkdir(parents=True, exist_ok=True)

USER_PDB = USER_INPUT_DIR / Path(original_name).name
USER_PDB.write_bytes(pdb_bytes)

print("Uploaded:", original_name)
print("Saved as:", USER_PDB)
print("Size:", f"{USER_PDB.stat().st_size / 1024:.1f} KiB")

Saving model_01 (2).pdb to model_01 (2).pdb
Uploaded: model_01 (2).pdb
Saved as: /content/LigandMPNN/user_inputs/model_01 (2).pdb
Size: 367.2 KiB


In [ ]:
# 9. 快速检查 PDB 中的链与 HETATM
from collections import defaultdict

chain_residues = defaultdict(set)
hetero_resnames = set()

with USER_PDB.open("r", encoding="utf-8", errors="replace") as handle:
    for line in handle:
        record = line[:6].strip()

        if record == "ATOM":
            chain = line[21].strip() or "_"
            resseq = line[22:26].strip()
            icode = line[26].strip()
            resname = line[17:20].strip()
            chain_residues[chain].add((resseq, icode, resname))

        elif record == "HETATM":
            hetero_resnames.add(line[17:20].strip())

print("Protein chains:")
if chain_residues:
    for chain, residues in sorted(chain_residues.items()):
        print(f"  chain {chain}: {len(residues)} residues")
else:
    print("  未识别到 ATOM 记录。")

print("\nHETATM residue names:")
print(" ", sorted(hetero_resnames) if hetero_resnames else "None")

Protein chains:
  chain B: 601 residues

HETATM residue names:
  None


In [ ]:
# 10. 设置自己的设计参数
#
# 含配体/金属/辅因子上下文：ligand_mpnn
# 纯蛋白或 binder 设计：protein_mpnn
MODEL_TYPE = "protein_mpnn"

# 多链用逗号分隔，例如 "A,B"。
# 空字符串表示设计所有解析到的蛋白链。
CHAINS_TO_DESIGN = "A"

SEED = 112
TEMPERATURE = 0.10
BATCH_SIZE = 5
NUMBER_OF_BATCHES = 10
PARSE_ATOMS_WITH_ZERO_OCCUPANCY = 1

# 固定残基示例："A12 A13 A25"
# 仅在所选设计链内固定这些残基；空字符串表示不额外固定。
FIXED_RESIDUES = ""

# 只重设计这些残基示例："A12 A13 A25"
# 与 FIXED_RESIDUES 二选一；空字符串表示不启用局部重设计。
REDESIGNED_RESIDUES = ""

# 输出统计 pt 文件。
SAVE_STATS = 1

if FIXED_RESIDUES.strip() and REDESIGNED_RESIDUES.strip():
    raise ValueError(
        "FIXED_RESIDUES 与 REDESIGNED_RESIDUES 不应同时使用。"
    )

if MODEL_TYPE not in {"ligand_mpnn", "protein_mpnn"}:
    raise ValueError(
        "MODEL_TYPE 只能是 ligand_mpnn 或 protein_mpnn。"
    )

print("Model:", MODEL_TYPE)
print("Chains to design:", CHAINS_TO_DESIGN or "all protein chains")
print("Total generated sequences:",
      BATCH_SIZE * NUMBER_OF_BATCHES)

Model: protein_mpnn
Chains to design: A
Total generated sequences: 50


In [ ]:
# 11. 运行自己的设计任务
import shutil
import sys
from pathlib import Path

USER_OUT = ROOT / "outputs" / "user_design"
shutil.rmtree(USER_OUT, ignore_errors=True)

if MODEL_TYPE == "ligand_mpnn":
    checkpoint_flag = "--checkpoint_ligand_mpnn"
    checkpoint_path = WEIGHTS["ligand_mpnn_010"]["path"]
else:
    checkpoint_flag = "--checkpoint_protein_mpnn"
    checkpoint_path = WEIGHTS["protein_mpnn_020"]["path"]

user_command = [
    sys.executable, "-u", "run.py",
    "--model_type", MODEL_TYPE,
    checkpoint_flag, checkpoint_path,
    "--seed", str(SEED),
    "--pdb_path", USER_PDB,
    "--out_folder", USER_OUT,
    "--batch_size", str(BATCH_SIZE),
    "--number_of_batches", str(NUMBER_OF_BATCHES),
    "--temperature", str(TEMPERATURE),
    "--parse_atoms_with_zero_occupancy",
    str(PARSE_ATOMS_WITH_ZERO_OCCUPANCY),
    "--save_stats", str(SAVE_STATS),
    "--verbose", "1",
]

if CHAINS_TO_DESIGN.strip():
    user_command.extend(
        ["--chains_to_design", CHAINS_TO_DESIGN.strip()]
    )

if FIXED_RESIDUES.strip():
    user_command.extend(
        ["--fixed_residues", FIXED_RESIDUES.strip()]
    )

if REDESIGNED_RESIDUES.strip():
    user_command.extend(
        ["--redesigned_residues", REDESIGNED_RESIDUES.strip()]
    )

run_and_show(user_command)

Running command:

/usr/bin/python3 -u run.py --model_type protein_mpnn --checkpoint_protein_mpnn /content/LigandMPNN/model_params/proteinmpnn_v_48_020.pt --seed 112 --pdb_path /content/LigandMPNN/user_inputs/model_01 (2).pdb --out_folder /content/LigandMPNN/outputs/user_design --batch_size 5 --number_of_batches 10 --temperature 0.1 --parse_atoms_with_zero_occupancy 1 --save_stats 1 --verbose 1 --chains_to_design A

Designing protein from this path: /content/LigandMPNN/user_inputs/model_01 (2).pdb
These residues will be redesigned:  []
These residues will be fixed:  ['B38', 'B39', 'B40', 'B41', 'B42', 'B43', 'B44', 'B45', 'B46', 'B47', 'B48', 'B49', 'B50', 'B51', 'B52', 'B53', 'B54', 'B55', 'B56', 'B57', 'B58', 'B59', 'B60', 'B61', 'B62', 'B63', 'B64', 'B65', 'B66', 'B67', 'B68', 'B69', 'B70', 'B71', 'B72', 'B73', 'B74', 'B75', 'B76', 'B77', 'B78', 'B79', 'B80', 'B81', 'B82', 'B83', 'B84', 'B85', 'B86', 'B87', 'B88', 'B89', 'B90', 'B91', 'B92', 'B93', 'B94', 'B95', 'B96', 'B97', 'B98', 

CompletedProcess(args=['/usr/bin/python3', '-u', 'run.py', '--model_type', 'protein_mpnn', '--checkpoint_protein_mpnn', '/content/LigandMPNN/model_params/proteinmpnn_v_48_020.pt', '--seed', '112', '--pdb_path', '/content/LigandMPNN/user_inputs/model_01 (2).pdb', '--out_folder', '/content/LigandMPNN/outputs/user_design', '--batch_size', '5', '--number_of_batches', '10', '--temperature', '0.1', '--parse_atoms_with_zero_occupancy', '1', '--save_stats', '1', '--verbose', '1', '--chains_to_design', 'A'], returncode=0, stdout="Designing protein from this path: /content/LigandMPNN/user_inputs/model_01 (2).pdb\nThese residues will be redesigned:  []\nThese residues will be fixed:  ['B38', 'B39', 'B40', 'B41', 'B42', 'B43', 'B44', 'B45', 'B46', 'B47', 'B48', 'B49', 'B50', 'B51', 'B52', 'B53', 'B54', 'B55', 'B56', 'B57', 'B58', 'B59', 'B60', 'B61', 'B62', 'B63', 'B64', 'B65', 'B66', 'B67', 'B68', 'B69', 'B70', 'B71', 'B72', 'B73', 'B74', 'B75', 'B76', 'B77', 'B78', 'B79', 'B80', 'B81', 'B82', 'B

In [ ]:
# 12. 显示用户任务的 FASTA
user_fastas = sorted((USER_OUT / "seqs").glob("*.fa"))

if not user_fastas:
    raise FileNotFoundError(
        f"没有在 {USER_OUT / 'seqs'} 中找到 FASTA。"
    )

for fasta in user_fastas:
    print("\n" + "=" * 90)
    print(fasta.name)
    print("=" * 90)
    print(fasta.read_text(encoding="utf-8"))


model_01 (2).fa
>model_01 (2), T=0.1, seed=112, num_res=0, num_ligand_res=0, use_ligand_context=True, ligand_cutoff_distance=8.0, batch_size=5, number_of_batches=10, model_path=/content/LigandMPNN/model_params/proteinmpnn_v_48_020.pt
SDMSHVIITETHSTGLRLDQGAGDYYWSEMPSRVTQLHNNDPNRVVLTEIEFSDGSRHMLSGMSMGVGAKAYGIINPQIMSQGGLKTQITASADLSLDVGYFNTGTSGTIPQKLRDGTGCQHMFGAFSGRRGFASSAMYLGGAALYKSAWSGSGYVVADAGTLTIPSDYVRHPGARNFGFNAIYVRGRSCNRVLYGMEGPNYTTGGAVQGASSSGALNFTYNPSNPESPKYSVGFARADPTNYAYWESMGDPNDSANGPIGIYSEHLGIYPSKITWYVTNLVYNGSGYNIDGGLFNGNDIKLSPREFIIKGVNVNNTSWKFINFIEKNFNVGNRADFRDVGCNLSKDSPSTGISGIATFGLPTTESNNAPSIKGGNVGGLHANVVSIYNFLPSASWYVSSNPPKIGNNYGDVWSENLLPLRLLGGSGSTILSGNIVFQGNGSVHVGTVGLDLNSSRNGAIVCTMEFIDDTWLSAGGIGCFNPTEMLSQGAEYGDSRFRIGGNTINKKLHQILSLPAGEYVPFFTIKGTVVNACKLQAAAYNPTPYWVSGLPGSVGQTGYYTLTYYMRNDGNNNISIWLDSSMSNIIGMKACLPNIKLIIQR
>model_01 (2), id=1, T=0.1, seed=112, overall_confidence=1.0000, ligand_confidence=1.0000, seq_rec=nan
SDMSHVIITETHSTGLRLDQGAGDYYWSEMPSRVTQLHNNDPNRVVLTEIEFSDGSRHML

In [ ]:
# 13. 可选：读取 stats 文件并显示基础字段
import torch

stats_files = sorted((USER_OUT / "stats").glob("*.pt"))

if not stats_files:
    print("没有 stats 文件；请确认 SAVE_STATS = 1。")
else:
    for stats_path in stats_files:
        stats = torch.load(
            stats_path,
            map_location="cpu",
            weights_only=False,
        )
        print("\n", stats_path.name)
        print(" fields:", sorted(stats.keys()))
        print(" seed:", stats.get("seed"))
        print(" temperature:", stats.get("temperature"))

        generated = stats.get("generated_sequences")
        if generated is not None:
            print(" generated_sequences shape:",
                  tuple(generated.shape))


 model_01 (2).pt
 fields: ['chain_mask', 'decoding_order', 'generated_sequences', 'log_probs', 'mask', 'native_sequence', 'sampling_probs', 'seed', 'temperature']
 seed: 112
 temperature: 0.1
 generated_sequences shape: (50, 601)


In [ ]:
# 14. 打包并下载全部用户结果
from google.colab import files
import shutil
from pathlib import Path

archive_base = Path("/content/ligandmpnn_user_design")
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(USER_OUT),
    )
)

print("Archive:", archive_path)
print("Size:", f"{archive_path.stat().st_size / 1024**2:.2f} MiB")

files.download(str(archive_path))

Archive: /content/ligandmpnn_user_design.zip
Size: 2.06 MiB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 常用参数示例

### 1. RFdiffusion binder，只设计 A 链

```python
MODEL_TYPE = "protein_mpnn"
CHAINS_TO_DESIGN = "A"
BATCH_SIZE = 5
NUMBER_OF_BATCHES = 10
```

### 2. 含配体的酶口袋设计

```python
MODEL_TYPE = "ligand_mpnn"
CHAINS_TO_DESIGN = "A"
```

### 3. 固定催化残基

```python
FIXED_RESIDUES = "A45 A102 A178"
REDESIGNED_RESIDUES = ""
```

### 4. 只设计少数位置

```python
FIXED_RESIDUES = ""
REDESIGNED_RESIDUES = "A45 A46 A49 A102"
```

### 5. 使用其它官方 checkpoint

官方下载脚本还提供 ProteinMPNN `002/010/020/030` 与 LigandMPNN `005/010/020/030` checkpoint。  
下载相应 `.pt` 后，只需修改 `checkpoint_path`。

## 故障排查

### 出现 `Weights only load failed`

重新运行“修复 PyTorch 2.6+ checkpoint 加载兼容性”单元格，然后再运行任务。

### 只看到 `CalledProcessError`

本 Notebook 的 `run_and_show()` 会把 stdout 和 stderr 合并显示。复制上方真正的 Python traceback，而不是只复制最后一行。

### 找不到 ligand atoms

检查输入 PDB 是否含 `HETATM`，以及需要的小分子、金属或辅因子是否在文件中。纯蛋白体系通常改用 `protein_mpnn`。

### 链没有被设计

检查 PDB 的真实 chain ID，并确认 `CHAINS_TO_DESIGN` 使用逗号分隔，例如 `"A,B"`。

### Colab 重启后文件消失

`/content` 是临时存储。重启运行时后需要重新运行安装单元，并重新上传 PDB。